# Nye FPL-snapshots og frosne prediksjoner

Kjør denne notebooken for å lese siste lokale innsamling. Ingen API-kall eller modelltrening skjer her.

Innsamleren kjører hver sjette time som en lokal macOS LaunchAgent. Maskinen må være på og brukeren innlogget; søvn/offline kan gi hull. Snapshotet er tatt på innsamlingstidspunktet, og er ikke nødvendigvis siste informasjon rett før deadline.

Prediksjonene bruker eksisterende ensemble, trent på kampbaserte historiske features. De er **eksperimentelle**. Nye snapshots bygger tidsstemplet grunnlag for fremtidig evaluering, men oppgraderer ikke den gamle treningen til en verifisert deadline-backtest.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/capture_fpl.py').exists())
parent=ROOT/'data/raw/live_fpl'
captures=[]
for path in sorted(parent.glob('capture_*/manifest.json')):
    m=json.loads(path.read_text())
    captures.append({'directory':str(path.parent), 'started_at':m['started_at'],
        'status':m['status'],'forecast_status':m.get('forecast_status'),
        'season':m.get('season'),'GW':(m.get('target_event') or {}).get('id'),
        'deadline':(m.get('target_event') or {}).get('deadline_time'),
        'forecast_at':m.get('forecast_at'),'players':m.get('captured_players'),
        'error':m.get('error')})
status=pd.DataFrame(captures)
display(status.drop(columns='directory').tail(12))

## Siste gyldig frosne prediksjon

Rangeringen nedenfor har ingen budsjett- eller troppsbegrensning. Spillere uten planlagt kamp får null. Bruk tidsstempelet for å se hvor gammel informasjonen er.

In [ ]:
valid=status[status.forecast_status.eq('experimental_frozen')].sort_values('forecast_at')
if len(valid):
    selected=valid.iloc[-1]
    forecast=pd.read_csv(Path(selected.directory)/'forecast.csv')
    print('Innsamling:',selected.started_at,'Deadline:',selected.deadline)
    display(forecast.head(25).round(3))
else:
    print('Ingen gyldige frosne prediksjoner ennå.')

## Evaluering når runden er ferdig

Innsamleren lagrer utfall først når FPL markerer runden som ferdig og data-kontrollert. Senere korrigerte utfall lagres som nye observasjoner. Til sammenligningen nedenfor brukes siste gyldige prediksjon før deadline per gameweek og sist observerte fasit. Ingen visekaptein eller chips.

In [ ]:
observations=[]
for path in sorted(parent.glob('capture_*/observed_outcomes.csv')):
    observations.append(pd.read_csv(path))
if observations:
    observed=pd.concat(observations).sort_values('labels_observed_at').drop_duplicates(
        ['forecast_path','player_id'],keep='last')
    final_forecasts=valid.sort_values('forecast_at').drop_duplicates(['season','GW'],keep='last')
    paths={str(Path(p)/'forecast.csv') for p in final_forecasts.directory}
    evaluated=observed[observed.forecast_path.isin(paths)].copy()
    evaluated['abs_error']=(evaluated.prediction-evaluated.actual_points).abs()
    display(evaluated.groupby(['season','GW']).agg(n=('player_id','size'),
        MAE=('abs_error','mean'),actual_mean=('actual_points','mean'),
        predicted_mean=('prediction','mean')).round(3))
else:
    print('Ingen fullførte runder med egne forhåndslagrede prediksjoner ennå.')

## Historisk pilot: DGW25 i 2024–25

Piloten undersøkte historiske Git-versjoner før deadline. Rapporten skiller kandidatkilder fra dokumentert tilgjengelighet. En unsigned commitdato og innhentingskode er ikke et selvstendig capture-tidsstempel.

In [ ]:
reports=list((ROOT/'data/raw/historical_pilot').glob('dgw25_*/report.json'))
if reports:
    p=max(reports,key=lambda x:x.stat().st_mtime)
    pilot=json.loads(p.read_text())
    print('Godkjent:',pilot['approved'])
    print(pilot['conclusion'])
    display(pd.DataFrame(pilot['candidate_sources']))
print('Drift og stopp/start: notebooks/LIVE_COLLECTION.md')